# Практическая работа № 2. Нейронные сети
## Прогнозирование стоимости жилья

**Дисциплина:** Прикладной искусственный интеллект.

**Выполнил(а):** ФИО, ИТМО ID.

Авторы практикума: Ким Станислав Александрович, Евстафьев Олег Александрович.

Получите исходные `train.csv`, `test.csv`, `data_description.txt` на сайте курса. Они не включены в этот ноутбук. Здесь нет заранее вычисленных результатов.

Самостоятельная часть: исследуйте архитектуру, эпохи, размер пакета, оптимизатор и функцию потерь по плану методички, затем объясните наблюдения.

**Первое знакомство с нейросетью.** Сначала проследите путь одного дома: строка таблицы → подготовленные числа → сеть → прогноз цены. До изменения настроек выполните исходный вариант B0 и объясните его MAE в денежных единицах.

Веса находятся при обучении; ширина слоёв, число эпох и размер пакета задаются до него. `run_experiment` — функция из обычного файла `code/lab2_utils.py`; её код можно открыть и прочитать.

Выполняйте ячейки сверху вниз. После изменения исходных данных или перед сдачей перезапустите среду и выполните всё заново. Для запуска скачайте весь проект: учебные модули находятся в общей папке `code`, а данные — в папке `data` рядом с этим ноутбуком.

## 1. Среда и файлы
Данные жилья поместите в папку `data` рядом с этим ноутбуком. Первая ячейка подключит общие модули из корня проекта и выберет каталог второй работы. Файлы первой работы сюда не подходят.

In [ ]:
from pathlib import Path
import sys
import os
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (CURRENT_DIR, *CURRENT_DIR.parents)
     if (p / "code").is_dir() and (p / "labs").is_dir()), None
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open the complete practicum project with code/ and labs/")
LAB_DIR = PROJECT_ROOT / "labs" / "02_regression"
sys.path.insert(0, str(PROJECT_ROOT / "code"))
os.chdir(LAB_DIR)
print("Working directory:", LAB_DIR)
import importlib.metadata as metadata
import numpy as np
import pandas as pd
from lab2_utils import load_course_data, constant_baselines
from lab2_utils import run_experiment, plot_diagnostics, save_experiment

for name in ("numpy", "pandas", "scikit-learn", "keras"):
    print(name, metadata.version(name))
DATA_DIR = Path("data")

## 2. Изучение данных
Зафиксируйте размеры таблиц, виды признаков и смысл пропусков. По словарю проверьте категориальные признаки, записанные числовыми кодами. `Id` — идентификатор, `SalePrice` — целевая переменная.

In [ ]:
train_data = pd.read_csv(DATA_DIR / "train.csv")
test_data = pd.read_csv(DATA_DIR / "test.csv")
print(train_data.shape, test_data.shape)
display(train_data.head())
display(test_data.head())
display(train_data.isna().sum().sort_values(ascending=False).head(10))

**Ваши наблюдения:** опишите признаки, пропуски и проверки схемы данных.

## 3. Предобработка без утечки
Разделение сырых размеченных строк — 80/20, seed 40. Статистики и кодировщик обучаются только на обучающей части. Цена стандартизуется отдельно и возвращается к исходным единицам при оценке. Прочитайте `prepare_frames` в `lab2_utils.py` и объясните её шаги.

In [ ]:
data = load_course_data(DATA_DIR, seed=40)
print("Shapes:", data["Atr"].shape, data["Ava"].shape, data["Atest"].shape)
print("Removed:", data["removed_columns"])
print("First feature names:", data["feature_names"][:10])
assert not set(data["train_ids"]) & set(data["validation_ids"])

## 4. Константные базовые прогнозы
Оба прогноза используют статистики обучающих цен. MAE и RMSE считаются по известным ценам валидации.

In [ ]:
baseline_table = pd.DataFrame(constant_baselines(data)).T
display(baseline_table)

## 5. Исходная нейросеть
Исследуйте исходную реализацию `run_experiment`: постройте схему слоёв и проверьте число параметров вручную. Следующая ячейка выполняет один исходный запуск; она не запускает весь план исследований.

In [ ]:
result = run_experiment(
    data, hidden=(64,), epochs=100, batch_size=32,
    optimizer="adam", learning_rate=1e-3, loss="mse",
    seed=40, early_stop=False, verbose=0,
)
result["model"].summary()
display(pd.DataFrame([result["metrics"]]))

## 6. Диагностика
Сопоставьте кривые обучения и валидации, фактические и предсказанные цены, остатки. При стандартизованной цели поле `loss` выражено в её масштабе; график ниже пересчитан в единицы цены.

In [ ]:
fig = plot_diagnostics(data, result)
validation = pd.DataFrame({
    "Id": data["validation_ids"],
    "SalePrice": data["yva"],
    "Prediction": result["validation_predictions"],
})
validation["AbsoluteError"] = abs(validation["SalePrice"] - validation["Prediction"])
display(validation.nlargest(5, "AbsoluteError"))

**Ваш анализ:** объясните крупные ошибки и поведение кривых. Не удаляйте неудачные объекты ради улучшения оценки.

## 7. План самостоятельных опытов
B0: (64,), 100 эпох, пакет 32, Adam, MSE.

Измените по одному фактору:
- W1/W2: скрытый слой 32/128;
- D1: скрытые слои (64, 32);
- E1/E2: 30/200 эпох;
- P1/P2: пакет 16/64;
- O1: SGD;
- L1: MAE.

Скорость обучения 0.001; разбиение и seed 40 сохраняются. В этих опытах ранний останов выключен. Для каждого варианта создавайте новую модель и сохраняйте отдельную папку. Выберите конфигурацию по MAE валидации, дополнительно рассмотрите RMSE и сложность.

In [ ]:
experiment_rows = []
experiment_rows.append({"experiment": "B0", **result["config"], **result["metrics"]})
display(pd.DataFrame(experiment_rows))
# Add the other experiments and your explanations here.

## 8. Сохранение выбранного запуска
Замените `baseline_01` уникальным именем. Уже существующая папка не перезаписывается. `predictions_raw.csv` сохраняет прогноз без скрытой постобработки. Проверьте отрицательные цены и обоснуйте любые изменения на валидации.

In [ ]:
OUT_DIR = Path("runs") / "baseline_01"
save_experiment(data, result, OUT_DIR)
fig.savefig(OUT_DIR / "diagnostics.pdf", bbox_inches="tight")
pd.DataFrame(experiment_rows).to_csv(OUT_DIR / "experiment_table.csv", index=False)
output = pd.DataFrame({"Id": data["prediction_ids"], "SalePrice": result["predictions"]})
assert output["Id"].tolist() == test_data["Id"].tolist()
assert np.isfinite(output["SalePrice"]).all()
print("Negative predictions:", int((output["SalePrice"] < 0).sum()))
display(output.head())

## 9. Вывод и ответы на вопросы
Сформулируйте влияние каждого фактора по своим результатам. Объясните эпоху и итерацию, функции активации, MSE и MAE, роль валидации и ограничения полученной оценки. Полный перечень вопросов находится в методичке.

Не называйте метрики валидации оценкой на `test.csv`: правильные цены этого файла неизвестны.

## Дополнительно: среднее трёх прогнозов

Необязательное задание к обзору TabM и TabPFN-3. Мы обучаем три отдельные MLP, а не реализуем TabM. Один объект `data` сохраняет разбиение и предобработку; меняется только seed обучения. Усредняются прогнозы для одинаковых домов, а не значения MAE.

По умолчанию опыт выключен. Для выполнения сначала завершите основную работу, затем задайте `RUN_OPTIONAL_ENSEMBLE = True`. Это ещё три обучения B0. Заранее выберите новый каталог для результатов; существующие опыты не перезаписываются.

In [ ]:
RUN_OPTIONAL_ENSEMBLE = False

if RUN_OPTIONAL_ENSEMBLE:
    from lab2_utils import regression_metrics
    ensemble_dir = Path("runs/optional_ensemble_01")
    if ensemble_dir.exists():
        raise FileExistsError("Choose a new ensemble output directory")
    individual_predictions = []
    rows = []
    for init_seed in (40, 41, 42):
        trial = run_experiment(
            data, hidden=(64,), epochs=100, batch_size=32,
            optimizer="adam", learning_rate=1e-3, loss="mse",
            seed=init_seed, early_stop=False,
        )
        save_experiment(data, trial, ensemble_dir / f"seed_{init_seed}")
        individual_predictions.append(trial["validation_predictions"].copy())
        rows.append({"model": f"MLP seed {init_seed}", **trial["metrics"]})
    mean_prediction = np.stack(individual_predictions).mean(axis=0)
    rows.append({"model": "Average", **regression_metrics(data["yva"], mean_prediction)})
    comparison = pd.DataFrame(rows)
    comparison.to_csv(ensemble_dir / "comparison.csv", index=False)
    pd.DataFrame({
        "Id": data["validation_ids"], "SalePrice": data["yva"],
        "Prediction": mean_prediction,
    }).to_csv(ensemble_dir / "average_validation_predictions.csv", index=False)
    display(comparison)
else:
    print("Optional experiment is disabled; the core work is unchanged.")


**Ваш вывод:** для каких домов усреднение помогло? Как изменились MAE, RMSE и затраты на обучение? Почему это не проверка TabM или TabPFN-3 и не гарантированный интервал цены?